# PA2 — resumable SFT + DPO pipeline

Set `STAGE` below. Runs on Kaggle workers (survive laptop sleep).
If an input dataset `pa2-checkpoints` is attached, it is copied into `outputs/` first.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

# all | sft | dpo_hh | dpo_uf | lora | generate | judge | resume_dpo
STAGE = "judge"
REPO_URL = "https://github.com/RifaSalman/CS5326.git"
CLONE_DIR = Path("/kaggle/working/CS5326")
REPO_DIR = CLONE_DIR / "PA2"
print("STAGE", STAGE)

def run(cmd, check=True):
    print("$", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=check)

if not CLONE_DIR.exists():
    run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)])
else:
    run(["git", "-C", str(CLONE_DIR), "pull"], check=False)
os.chdir(REPO_DIR)

run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.43.0,<5", "datasets>=2.18,<5", "huggingface_hub", "accelerate", "tqdm", "pyyaml", "sentencepiece", "safetensors", "pytest", "matplotlib", "peft>=0.11.0", "trl>=0.9.0"])
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])

import torch
print("cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count(), flush=True)
!nvidia-smi || true
run([sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"])


In [ ]:
import os, sys, shutil, subprocess, traceback
from pathlib import Path

os.environ["PYTHONPATH"] = str(Path.cwd())
Path("outputs").mkdir(exist_ok=True)

# Restore prior artifacts from an attached Kaggle dataset if present.
import zipfile
for cand in [Path("/kaggle/input/pa2-checkpoints"), Path("/kaggle/input/pa2-sft-outputs")]:
    if not cand.exists():
        continue
    print("Restoring from", cand, flush=True)
    # dir-mode zip datasets ship top-level archives like sft.zip / dpo_hh.zip
    for zpath in sorted(cand.glob("*.zip")):
        dest = Path("outputs") / zpath.stem
        print("Extracting", zpath.name, "->", dest, flush=True)
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(dest)
    for src in cand.rglob("*"):
        if src.is_file() and src.suffix.lower() != ".zip":
            rel = src.relative_to(cand)
            dst = Path("outputs") / rel
            if rel.parts and rel.parts[0] == "outputs":
                dst = Path(*rel.parts)
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists():
                shutil.copy2(src, dst)
    break

# Promote final -> best when best is missing (SFT and both DPO tracks).
for track in ["sft", "dpo_hh", "dpo_uf"]:
    best = Path(f"outputs/{track}/best")
    final = Path(f"outputs/{track}/final")
    if final.exists() and not (best / "config.json").exists():
        print(f"Promoting outputs/{track}/final -> outputs/{track}/best", flush=True)
        if best.exists():
            shutil.rmtree(best)
        shutil.copytree(final, best)

def py(*args):
    subprocess.run([sys.executable, *args], check=True)

def should(*names):
    return STAGE in names or STAGE == "all"

def archive():
    # Judge artifacts are small; avoid re-zipping multi-GB checkpoints.
    if STAGE == "judge":
        out = Path("/kaggle/working/pa2_eval")
        if out.exists():
            shutil.rmtree(out)
        shutil.copytree("outputs/eval", out, dirs_exist_ok=True)
        shutil.make_archive("/kaggle/working/pa2_eval", "zip", out)
        print("Wrote /kaggle/working/pa2_eval.zip", flush=True)
    else:
        shutil.make_archive("/kaggle/working/pa2_outputs", "zip", "outputs")
        print("Wrote /kaggle/working/pa2_outputs.zip", flush=True)


try:
    if should("sft"):
        py("scripts/train_sft.py", "--output_dir", "outputs/sft")
        py("scripts/plot_metrics.py", "--metrics_path", "outputs/sft/metrics.jsonl", "--output_dir", "report/figures/sft", "--title", "SFT")

    if should("dpo_hh", "resume_dpo") and not Path("outputs/dpo_hh/best/config.json").exists():
        py("scripts/train_dpo.py", "--track", "hh", "--sft_checkpoint", "outputs/sft/best", "--output_dir", "outputs/dpo_hh")
        py("scripts/plot_metrics.py", "--metrics_path", "outputs/dpo_hh/metrics.jsonl", "--output_dir", "report/figures/dpo_hh", "--title", "DPO HH")

    if should("dpo_uf", "resume_dpo") and not Path("outputs/dpo_uf/best/config.json").exists():
        py("scripts/train_dpo.py", "--track", "ultrafeedback", "--sft_checkpoint", "outputs/sft/best", "--output_dir", "outputs/dpo_uf")
        py("scripts/plot_metrics.py", "--metrics_path", "outputs/dpo_uf/metrics.jsonl", "--output_dir", "report/figures/dpo_uf", "--title", "DPO UltraFeedback")

    if should("lora", "resume_dpo", "lora_generate") and not (
        Path("outputs/lora_persona_dpo/best/adapter_config.json").exists()
        or Path("outputs/lora_persona_dpo/best/config.json").exists()
    ):
        py("scripts/train_lora_dpo.py", "--output_dir", "outputs/lora_persona_dpo")

    if should("generate", "resume_dpo", "lora_generate"):
        needed = [
            Path("outputs/sft/best/config.json"),
            Path("outputs/dpo_uf/best/config.json"),
            Path("outputs/dpo_hh/best/config.json"),
        ]
        missing = [str(p) for p in needed if not p.exists()]
        if missing:
            print("Skipping generate; missing checkpoints:", missing, flush=True)
        else:
            if not Path("eval_sets/alpaca_eval_300_seed42.json").exists():
                py("scripts/prepare_alpaca_eval.py", "--output", "eval_sets/alpaca_eval_300_seed42.json")
            py("scripts/generate.py", "--model_path", "outputs/sft/best", "--prompts_path", "eval_sets/alpaca_eval_300_seed42.json", "--output_path", "outputs/gen/sft_alpaca.json", "--generator_name", "sft")
            py("scripts/generate.py", "--model_path", "outputs/dpo_uf/best", "--prompts_path", "eval_sets/alpaca_eval_300_seed42.json", "--output_path", "outputs/gen/dpo_uf_alpaca.json", "--generator_name", "dpo_uf")
            py("scripts/generate.py", "--model_path", "outputs/sft/best", "--prompts_path", "eval_sets/hh_rlhf_eval_300_seed42.json", "--output_path", "outputs/gen/sft_hh.json", "--generator_name", "sft")
            py("scripts/generate.py", "--model_path", "outputs/dpo_hh/best", "--prompts_path", "eval_sets/hh_rlhf_eval_300_seed42.json", "--output_path", "outputs/gen/dpo_hh.json", "--generator_name", "dpo_hh")
            py("scripts/build_pairs.py", "--sft_path", "outputs/gen/sft_alpaca.json", "--dpo_path", "outputs/gen/dpo_uf_alpaca.json", "--output_path", "outputs/eval/dpo_uf_vs_sft_pairs.jsonl", "--sft_name", "sft", "--dpo_name", "dpo_uf")
            py("scripts/build_pairs.py", "--sft_path", "outputs/gen/sft_hh.json", "--dpo_path", "outputs/gen/dpo_hh.json", "--output_path", "outputs/eval/dpo_hh_vs_sft_pairs.jsonl", "--sft_name", "sft", "--dpo_name", "dpo_hh")

    if should("judge"):
        Path("outputs/eval").mkdir(parents=True, exist_ok=True)
        for name in ["dpo_uf_vs_sft_pairs.jsonl", "dpo_hh_vs_sft_pairs.jsonl"]:
            srcp = Path("eval_sets/pairs") / name
            dstp = Path("outputs/eval") / name
            if srcp.exists() and not dstp.exists():
                shutil.copy2(srcp, dstp)
                print("Seeded", dstp, flush=True)
        py("scripts/judge_pairwise_kaggle.py", "--pairs_path", "outputs/eval/dpo_uf_vs_sft_pairs.jsonl", "--output_path", "outputs/eval/dpo_uf_vs_sft_judged.jsonl", "--reuse_judgments")
        py("scripts/judge_pairwise_kaggle.py", "--pairs_path", "outputs/eval/dpo_hh_vs_sft_pairs.jsonl", "--output_path", "outputs/eval/dpo_hh_vs_sft_judged.jsonl", "--reuse_judgments", "--safety_focus")

except Exception:
    traceback.print_exc()
    raise
finally:
    archive()
    print("Done stage", STAGE, flush=True)
